session import and session buliding

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Cross_System_Monitoring")
    # Delta Lake Configurations
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.file.impl", "org.apache.hadoop.fs.local.LocalFs")
    # Network configurations to prevent Py4J Java timeouts
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "60s")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

reading the datasets

In [2]:
crm = spark.read.csv(
    "C:\Jatin\Celebal_internship\cross-system-monitoring\data\crm_dataset.csv",
    header=True,
    inferSchema=True
)

billing = spark.read.csv(
    "C:\Jatin\Celebal_internship\cross-system-monitoring\data\\billing_dataset.csv",
    header=True,
    inferSchema=True
)

analytics = spark.read.csv(
    "C:\Jatin\Celebal_internship\cross-system-monitoring\data\\analytics_dataset.csv",
    header=True,
    inferSchema=True
)

schema

In [3]:
print("CRM Schema")
crm.printSchema()

print("Billing Schema")
billing.printSchema()

print("Analytics Schema")
analytics.printSchema()

CRM Schema
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- city: string (nullable = true)

Billing Schema
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- status: string (nullable = true)

Analytics Schema
root
 |-- date: date (nullable = true)
 |-- total_customers: integer (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_transaction: double (nullable = true)



In [4]:
print("staring rows")
crm.show(5)

billing.show(5)

analytics.show(5)

staring rows
+-----------+-------------+--------------------+-----------+--------+
|customer_id|         name|               email|signup_date|    city|
+-----------+-------------+--------------------+-----------+--------+
|  CRM010304|   Rahul Bose|rahul.bose239@gma...| 2022-02-09|Vadodara|
|  CRM006432|  Deepa Kumar|deepa.kumar616@ya...| 2023-05-27| Lucknow|
|  CRM000305| Rahul Mishra|rahul.mishra260@h...| 2022-08-16| Kolkata|
|  CRM009603|Kritika Singh|kritika.singh830@...| 2024-01-04|   Patna|
|  CRM007384| Arjun Sharma|arjun.sharma411@y...| 2023-11-09|  Mumbai|
+-----------+-------------+--------------------+-----------+--------+
only showing top 5 rows

+--------------+-----------+-------+----------------+---------+
|transaction_id|customer_id| amount|transaction_date|   status|
+--------------+-----------+-------+----------------+---------+
|    TXN0004112|  CRM003884|2454.45|      2022-09-03|completed|
|    TXN0007685|  CRM008441|1353.66|      2023-11-26|  pending|
|    TXN0007

saving filter data into delta table

In [5]:
crm.write.format("delta").mode("overwrite").save("../bronze/crm")

billing.write.format("delta").mode("overwrite").save("../bronze/billing")

analytics.write.format("delta").mode("overwrite").save("../bronze/analytics")